[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_17_Advanced_Evals_RAGAS_LLM_Judge.ipynb)

# 🧪 Lesson 17: Advanced Evals — RAGAS + LLM-Judge Pipelines

**Course:** Learn AI → Phase 3: Production AI Engineering  
**Date:** 2026-05-16  
**Prerequisite:** Lesson 16 (Deployment — FastAPI + Docker)

---

## 🎯 What You'll Learn

By the end of this lesson you will:
1. Understand **why evals are the most important engineering discipline** in AI systems
2. Master the **RAGAS framework** for evaluating RAG pipelines (4 core metrics)
3. Build **LLM-as-Judge** — using a language model to score another model's output
4. Design a **complete eval pipeline** you can run in CI/CD
5. Run a **capstone eval** on the AutoResearcher agent you built in earlier lessons

---

## 🧠 Concept: Why Evals Matter More Than Anything Else

In traditional software:
- Unit tests tell you if your code **does what you wrote it to do**
- The behavior is deterministic — same input = same output

In AI systems:
- LLMs are **non-deterministic** — same input can produce different outputs
- You can't unit-test "was this response good?"
- A prompt change that "feels" better might actually degrade performance at scale

**Evals are the solution.** They let you:
- **Measure** quality objectively (not just feel it)
- **Detect regressions** when you change prompts, models, or data
- **Compare** model versions (Claude 3.5 vs Claude 4 for your use case)
- **Gate deployments** — only ship if eval score > threshold

> 💡 "Without evals, you're flying blind. With evals, you're flying with instruments."

---

## 🗺️ The Eval Landscape

| Type | What it measures | Example |
|------|-----------------|----------|
| **Functional evals** | Does the output have the right structure/format? | "Does the JSON parse?" |
| **Reference-based evals** | How close is the output to a known-good answer? | BLEU, ROUGE, exact match |
| **LLM-Judge evals** | Is the output good by human-like criteria? | Helpfulness, faithfulness, coherence |
| **RAG-specific evals** | Is the RAG pipeline retrieving and using context well? | RAGAS |
| **Human evals** | Actual human preference ratings | A/B testing, RLHF |

Today we focus on **LLM-Judge** and **RAGAS** — the two most powerful automated eval techniques.

---
## ⚙️ Setup — Install Packages & Load API Key

In [ ]:
# Install all dependencies
!pip install anthropic ragas datasets chromadb sentence-transformers langchain langchain-anthropic langchain-community pandas rich -q

print("✅ Packages installed")

In [ ]:
import os

# Load API key from Colab Secrets (one-time setup: Runtime → Secrets → add ANTHROPIC_API_KEY)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Running locally — set the key here or via .env
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"
    print("⚠️ Using fallback API key — replace with your key if running locally")

import anthropic
client = anthropic.Anthropic()
print("✅ Anthropic client ready")

---
# 🔬 Part 1: LLM-as-Judge — The Core Pattern

## What is LLM-as-Judge?

You use a **capable LLM** to evaluate the output of **another LLM** (or even the same one).

```
User Question + System Answer → [Judge LLM] → Score (0-10) + Reasoning
```

**Why does this work?**
- LLMs are excellent at following rubrics and criteria
- They can evaluate things humans care about (helpfulness, tone, accuracy) that are hard to measure with code
- Much cheaper and faster than human evaluation at scale
- Research (Stanford, Anthropic, OpenAI) shows LLM judges correlate strongly with human preference (0.8+ correlation on many tasks)

**Key design principle:** The judge prompt is a rubric. The clearer and more specific your rubric, the more reliable and consistent the judgment.

## The 3 Judge Patterns

| Pattern | When to use |
|---------|------------|
| **Pointwise** | Score one response on a scale (1-10) | 
| **Pairwise** | Compare two responses, pick the better one (A/B testing) |
| **Reference** | Compare response to a gold-standard reference answer |

In [ ]:
import json

# ─── POINTWISE LLM JUDGE ───────────────────────────────────────────────
# This is the fundamental building block of eval pipelines.

def llm_judge_pointwise(question: str, answer: str, criteria: dict) -> dict:
    """
    Evaluate a single answer on multiple criteria.
    
    criteria = {
        "criterion_name": "description of what to check"
    }
    Returns: {criterion: {score: int, reasoning: str}}
    """
    criteria_text = "\n".join([
        f"- **{name}**: {desc}"
        for name, desc in criteria.items()
    ])
    
    prompt = f"""You are an expert evaluator assessing the quality of an AI assistant's answer.

## Question asked:
{question}

## Answer to evaluate:
{answer}

## Evaluation Criteria (score each 1-10):
{criteria_text}

## Instructions:
For each criterion:
1. Give a score from 1 to 10 (1=terrible, 5=adequate, 10=excellent)
2. Give 1-2 sentences of reasoning

Respond in this exact JSON format:
{{
  "criterion_name": {{
    "score": <int 1-10>,
    "reasoning": "<your reasoning>"
  }}
}}

Replace criterion_name with each actual criterion name from the list above."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Use fast/cheap model for judging
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    
    return json.loads(raw)


# ─── TEST IT ────────────────────────────────────────────────────────────
question = "What is a transformer in machine learning?"

# Good answer
good_answer = """A transformer is a neural network architecture introduced in the 2017 paper 
'Attention Is All You Need'. It uses a mechanism called self-attention to process sequential 
data (like text) by weighing the importance of each word relative to all others in the sequence. 
Unlike RNNs, transformers process all tokens in parallel, making them much faster to train. 
They are the foundation of modern LLMs like GPT, BERT, and Claude."""

# Weak answer
weak_answer = "It's a type of AI model used in NLP tasks."

criteria = {
    "accuracy": "Is the technical information correct and precise?",
    "completeness": "Does the answer cover the key aspects of the concept?",
    "clarity": "Is the explanation clear and well-structured for a learner?"
}

print("📋 Evaluating GOOD answer...")
good_scores = llm_judge_pointwise(question, good_answer, criteria)
for criterion, result in good_scores.items():
    print(f"  {criterion}: {result['score']}/10 — {result['reasoning']}")

print("\n📋 Evaluating WEAK answer...")
weak_scores = llm_judge_pointwise(question, weak_answer, criteria)
for criterion, result in weak_scores.items():
    print(f"  {criterion}: {result['score']}/10 — {result['reasoning']}")

### 💡 Pairwise Judge — "Which is better?"

Pairwise judgment is more reliable than pointwise — it avoids "score calibration" problems (one judge giving 7/10 where another gives 9/10 for the same answer). You just ask: **A or B?**

This is how Anthropic runs RLHF and how many A/B tests are structured.

In [ ]:
def llm_judge_pairwise(question: str, answer_a: str, answer_b: str, criterion: str) -> dict:
    """
    Compare two answers and determine which is better.
    Returns: {winner: 'A'|'B'|'tie', reasoning: str}
    
    💡 EXPERIMENT: Try swapping A and B to check for position bias.
    A well-calibrated judge should pick the same winner regardless of order.
    """
    prompt = f"""You are an expert evaluator comparing two AI responses.

## Question:
{question}

## Response A:
{answer_a}

## Response B:
{answer_b}

## Evaluation Criterion:
{criterion}

Which response better satisfies the criterion? 
Respond in JSON: {{"winner": "A" or "B" or "tie", "reasoning": "your reasoning"}}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)


result = llm_judge_pairwise(
    question=question,
    answer_a=good_answer,
    answer_b=weak_answer,
    criterion="Overall helpfulness and educational value for a software engineer learning AI"
)

print(f"🏆 Winner: Response {result['winner']}")
print(f"📝 Reasoning: {result['reasoning']}")

# 💡 EXPERIMENT: Now try swapping A and B — does the judge still pick the same winner?
# result_swapped = llm_judge_pairwise(question, weak_answer, good_answer, criterion)
# print(f"Swapped winner: {result_swapped['winner']}")  # Should be 'A' (was 'B' before, now the good one)

---
# 🏭 Part 2: Building a Reusable Eval Pipeline

A real eval pipeline has these components:

```
Test Dataset (questions + expected answers)
       ↓
System Under Test (your agent/RAG/LLM)
       ↓
Judge (LLM-as-Judge or RAGAS)
       ↓
Metrics Report (pass rate, average score, regressions)
```

Let's build one.

In [ ]:
import pandas as pd
from typing import Callable
import time

# ─── EVAL DATASET ───────────────────────────────────────────────────────
# In production this would be loaded from a file or database.
# Each entry = one test case.

eval_dataset = [
    {
        "id": "q1",
        "question": "What is the difference between a vector embedding and a traditional keyword search?",
        "reference_answer": (
            "Keyword search finds documents containing exact or similar words (lexical matching). "
            "Vector embeddings convert text into numerical vectors in a high-dimensional space, "
            "where semantically similar texts are close together regardless of word overlap. "
            "Embeddings capture meaning; keyword search captures word frequency."
        ),
        "tags": ["rag", "embeddings"]
    },
    {
        "id": "q2",
        "question": "Explain the ReAct agent loop in one paragraph.",
        "reference_answer": (
            "ReAct (Reason + Act) is an agent loop where the LLM alternates between Thought "
            "(reasoning about what to do), Action (calling a tool), and Observation (receiving "
            "the tool's result). This cycle repeats until the agent decides it has enough "
            "information to produce a final answer. The Thought step grounds tool use in reasoning."
        ),
        "tags": ["agents", "react"]
    },
    {
        "id": "q3",
        "question": "What is LoRA fine-tuning and when should you use it instead of full fine-tuning?",
        "reference_answer": (
            "LoRA (Low-Rank Adaptation) inserts small trainable matrices into the frozen layers "
            "of a pre-trained model, reducing the number of trainable parameters by 100-1000x. "
            "Use LoRA when GPU memory is limited or you want fast, cheap fine-tuning. "
            "Full fine-tuning is better when you have abundant compute and need maximum adaptation."
        ),
        "tags": ["fine-tuning"]
    }
]

print(f"✅ Eval dataset loaded: {len(eval_dataset)} test cases")

In [ ]:
# ─── SYSTEM UNDER TEST ──────────────────────────────────────────────────
# This simulates your AI system. In real use, this would call your
# actual deployed agent, RAG pipeline, or API endpoint.

def system_under_test(question: str, quality: str = "normal") -> str:
    """
    The AI system being evaluated.
    quality='normal' = standard claude-haiku response
    quality='poor'   = deliberately bad system prompt to simulate regression
    
    💡 EXPERIMENT: Set quality='poor' and observe how eval scores drop!
    """
    if quality == "poor":
        system = "Answer all questions in exactly one sentence, no matter how complex the topic."
    else:
        system = (
            "You are an expert AI/ML tutor. Explain concepts clearly with technical precision, "
            "suitable for a software engineer learning AI. Use 3-5 sentences."
        )
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text.strip()


# Quick test
test_response = system_under_test(eval_dataset[0]["question"])
print("🤖 System Under Test sample output:")
print(test_response)

In [ ]:
# ─── EVAL RUNNER ────────────────────────────────────────────────────────

def run_eval_pipeline(
    dataset: list,
    system_fn: Callable,
    judge_criteria: dict,
    pass_threshold: float = 7.0,
    quality: str = "normal"
) -> pd.DataFrame:
    """
    Run the full eval pipeline:
    1. Get system answer for each question
    2. Judge each answer on all criteria
    3. Compute aggregate metrics
    """
    results = []
    
    for case in dataset:
        print(f"  Evaluating [{case['id']}] {case['question'][:60]}...")
        
        # Step 1: Get system answer
        answer = system_fn(case["question"], quality=quality)
        
        # Step 2: Judge it
        scores = llm_judge_pointwise(case["question"], answer, judge_criteria)
        
        # Step 3: Aggregate
        avg_score = sum(v["score"] for v in scores.values()) / len(scores)
        passed = avg_score >= pass_threshold
        
        row = {
            "id": case["id"],
            "question": case["question"][:80],
            "answer_preview": answer[:120] + "...",
            "avg_score": round(avg_score, 2),
            "passed": passed,
            "tags": ", ".join(case.get("tags", []))
        }
        # Add individual criterion scores
        for criterion, result in scores.items():
            row[f"score_{criterion}"] = result["score"]
        
        results.append(row)
        time.sleep(0.5)  # Rate limit courtesy pause
    
    return pd.DataFrame(results)


# Define evaluation criteria
judge_criteria = {
    "accuracy": "Is the technical content correct and free of errors?",
    "completeness": "Are the key aspects of the concept covered without critical omissions?",
    "clarity": "Is the explanation clear, well-structured, and appropriately technical for a software engineer?"
}

print("🚀 Running eval pipeline (normal quality)...")
results_normal = run_eval_pipeline(eval_dataset, system_under_test, judge_criteria, quality="normal")

print("\n📊 EVAL RESULTS — Normal Quality:")
print(results_normal[["id", "avg_score", "passed", "score_accuracy", "score_completeness", "score_clarity"]].to_string(index=False))
print(f"\n✅ Pass rate: {results_normal['passed'].mean():.0%}")
print(f"📈 Average score: {results_normal['avg_score'].mean():.2f}/10")

In [ ]:
# ─── REGRESSION DETECTION ───────────────────────────────────────────────
# Now run the SAME eval on the POOR quality system to detect the regression.
# This is exactly what you'd do in CI/CD before shipping a prompt change.

print("🚨 Running eval pipeline (POOR quality — simulates regression)...")
results_poor = run_eval_pipeline(eval_dataset, system_under_test, judge_criteria, quality="poor")

print("\n📊 EVAL RESULTS — Poor Quality (Regression):")
print(results_poor[["id", "avg_score", "passed", "score_accuracy", "score_completeness", "score_clarity"]].to_string(index=False))
print(f"\n❌ Pass rate: {results_poor['passed'].mean():.0%}")
print(f"📉 Average score: {results_poor['avg_score'].mean():.2f}/10")

# Comparison
print("\n" + "="*50)
print("📊 REGRESSION REPORT")
print("="*50)
delta = results_normal['avg_score'].mean() - results_poor['avg_score'].mean()
print(f"Normal system avg score:  {results_normal['avg_score'].mean():.2f}/10")
print(f"Poor system avg score:    {results_poor['avg_score'].mean():.2f}/10")
print(f"Score delta:              {delta:+.2f}")
if delta > 1.0:
    print("🚨 REGRESSION DETECTED — Do not deploy this change!")
else:
    print("✅ No significant regression detected")

---
# 🎯 Part 3: RAGAS — Evaluating RAG Pipelines

## What is RAGAS?

RAGAS (RAG Assessment) is an open-source framework specifically designed to evaluate Retrieval-Augmented Generation systems. It measures whether your RAG pipeline is:

1. **Retrieving the right context** (are the chunks relevant?)
2. **Using the context faithfully** (is the answer grounded in the retrieved text?)
3. **Answering the actual question** (is the answer relevant to what was asked?)

## The 4 Core RAGAS Metrics

| Metric | What it measures | Formula |
|--------|-----------------|--------|
| **Faithfulness** | Does the answer only contain claims supported by the retrieved context? | claims_in_context / total_claims |
| **Answer Relevancy** | How relevant is the answer to the original question? | uses LLM to generate back-questions |
| **Context Recall** | What fraction of the ground-truth answer can be attributed to the retrieved context? | attributed_sentences / total_sentences |
| **Context Precision** | Are the retrieved chunks actually useful, or is there noise? | useful_chunks / total_chunks |

### 🧠 Visualizing the metrics:

```
Question ──────────────────────────────────────────────────────┐
     │                                                          │
     ↓                                                          │
  Retriever  ──→  Context chunks ──→  Generator  ──→  Answer   │
                       │                               │        │
             Context Precision                Faithfulness      │
             Context Recall             Answer Relevancy ←──────┘
```

## RAGAS requires:
- `question` — what the user asked
- `answer` — what the RAG system generated
- `contexts` — the list of retrieved text chunks
- `ground_truth` — the known correct answer (for recall/precision)

In [ ]:
# ─── BUILD A TINY RAG SYSTEM TO EVALUATE ─────────────────────────────────
# We'll build a minimal RAG pipeline and then run RAGAS on it.

import chromadb
from chromadb.utils import embedding_functions

# Sample knowledge base — AI concepts
knowledge_base = [
    {
        "id": "doc1",
        "text": (
            "Vector embeddings are numerical representations of text in a high-dimensional space. "
            "Semantically similar texts are placed close together in this space. "
            "Embeddings are generated by encoder models trained on large text corpora. "
            "They are the foundation of modern semantic search and RAG systems."
        )
    },
    {
        "id": "doc2",
        "text": (
            "Keyword search uses lexical matching — it finds documents containing the exact words "
            "from the query. BM25 is the classic keyword search algorithm. "
            "It is fast and explainable but fails on synonyms and conceptual queries."
        )
    },
    {
        "id": "doc3",
        "text": (
            "The ReAct framework interleaves reasoning and acting in LLM agents. "
            "The agent produces a Thought (what to do), an Action (tool call), "
            "and then observes the result before producing the next Thought. "
            "This loop continues until the agent has enough info to answer."
        )
    },
    {
        "id": "doc4",
        "text": (
            "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. "
            "Instead of updating all model weights, it inserts small trainable rank-decomposition matrices "
            "into the model's layers. This reduces memory requirements by 3-10x compared to full fine-tuning."
        )
    },
    {
        "id": "doc5",
        "text": (
            "Hallucination in LLMs refers to generating confident-sounding text that is factually wrong "
            "or not grounded in any provided context. RAG systems can reduce hallucination by grounding "
            "answers in retrieved documents, but the model can still ignore the context."
        )
    }
]

# Set up ChromaDB with sentence-transformers embeddings
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Delete collection if it exists (for clean re-runs)
try:
    chroma_client.delete_collection("ai_knowledge")
except:
    pass

collection = chroma_client.create_collection("ai_knowledge", embedding_function=ef)
collection.add(
    ids=[doc["id"] for doc in knowledge_base],
    documents=[doc["text"] for doc in knowledge_base]
)

print(f"✅ Knowledge base loaded: {len(knowledge_base)} documents in ChromaDB")


def rag_answer(question: str, n_results: int = 2) -> tuple[str, list[str]]:
    """
    Simple RAG pipeline: retrieve → generate.
    Returns (answer, retrieved_contexts)
    """
    # Retrieve
    results = collection.query(query_texts=[question], n_results=n_results)
    contexts = results["documents"][0]
    
    # Generate
    context_text = "\n\n".join(f"[Context {i+1}]:\n{ctx}" for i, ctx in enumerate(contexts))
    prompt = f"""Answer the question based ONLY on the provided context. 
If the context doesn't contain enough information, say so.

{context_text}

Question: {question}"""
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip(), contexts


# Quick test
ans, ctx = rag_answer("What is the difference between keyword search and vector embeddings?")
print("\n🔍 RAG Answer:")
print(ans)
print(f"\n📄 Retrieved {len(ctx)} context(s)")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

# ─── RAGAS EVAL DATASET ─────────────────────────────────────────────────
# RAGAS needs: question, answer, contexts, ground_truth

ragas_test_cases = [
    {
        "question": "What is the difference between keyword search and vector embeddings?",
        "ground_truth": (
            "Keyword search uses lexical matching to find documents containing exact query words. "
            "Vector embeddings represent text numerically in high-dimensional space where similar "
            "texts cluster together, enabling semantic search beyond exact word matching."
        )
    },
    {
        "question": "How does the ReAct agent loop work?",
        "ground_truth": (
            "ReAct agents alternate between Thought (reasoning what to do next), "
            "Action (executing a tool call), and Observation (receiving the result). "
            "This loop continues until the agent can produce a final answer."
        )
    },
    {
        "question": "What is LoRA and why use it over full fine-tuning?",
        "ground_truth": (
            "LoRA inserts small trainable rank-decomposition matrices into frozen model layers, "
            "reducing memory requirements by 3-10x vs full fine-tuning. "
            "Use it when GPU memory is limited or you want faster, cheaper adaptation."
        )
    }
]

# Collect RAG outputs for all test cases
print("🔄 Running RAG pipeline on test cases...")
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

for case in ragas_test_cases:
    answer, contexts = rag_answer(case["question"])
    ragas_data["question"].append(case["question"])
    ragas_data["answer"].append(answer)
    ragas_data["contexts"].append(contexts)
    ragas_data["ground_truth"].append(case["ground_truth"])
    print(f"  ✅ {case['question'][:60]}...")

ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\n✅ RAGAS dataset prepared: {len(ragas_dataset)} examples")

In [ ]:
# ─── RUN RAGAS ───────────────────────────────────────────────────────────
# RAGAS uses an LLM internally to compute the metrics.
# By default it uses OpenAI — we configure it to use Claude via LangChain.

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

# Configure RAGAS to use Claude as the judge LLM
judge_llm = ChatAnthropic(model="claude-haiku-4-5-20251001", anthropic_api_key=os.environ["ANTHROPIC_API_KEY"])
ragas_llm = LangchainLLMWrapper(judge_llm)

# Use free local embeddings (HuggingFace) instead of OpenAI embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

metrics = [faithfulness, answer_relevancy, context_recall, context_precision]

# Configure each metric to use our LLM + embeddings
for metric in metrics:
    metric.llm = ragas_llm
    if hasattr(metric, 'embeddings'):
        metric.embeddings = ragas_embeddings

print("🚀 Running RAGAS evaluation (this takes ~1-2 minutes)...")
ragas_results = evaluate(ragas_dataset, metrics=metrics)

print("\n📊 RAGAS RESULTS")
print("="*50)
for metric_name, score in ragas_results.items():
    bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
    print(f"  {metric_name:<22} {bar} {score:.3f}")

print("\n💡 Score interpretation:")
print("  > 0.8 = Good | 0.6-0.8 = Acceptable | < 0.6 = Needs work")

### 🧠 Reading Your RAGAS Scores

| Score | Diagnosis | Fix |
|-------|-----------|-----|
| **Low Faithfulness** | Model is hallucinating or ignoring context | Strengthen system prompt: "ONLY use context", add hallucination guardrails |
| **Low Answer Relevancy** | Answer drifts from the question | Add re-ranker, improve generation prompt |
| **Low Context Recall** | Retriever isn't finding the right chunks | Improve chunking strategy, try hybrid search |
| **Low Context Precision** | Too much irrelevant noise in retrieved chunks | Reduce `n_results`, add re-ranker, improve embeddings |

---

---
# 🔧 Part 4: Custom Faithfulness Checker (Manual RAGAS-style)

If RAGAS has dependency issues or you want full control, you can implement faithfulness checking yourself using LLM-as-Judge. This is how RAGAS works internally anyway.

**Algorithm:**
1. Extract all factual claims from the answer
2. For each claim, check if it's supported by the context
3. Faithfulness = supported_claims / total_claims

In [ ]:
def check_faithfulness(answer: str, contexts: list[str]) -> dict:
    """
    Manual faithfulness check — extracts claims and verifies each against context.
    This mirrors what RAGAS does internally.
    
    💡 EXPERIMENT: Try adding a fabricated fact to the answer and see if it's caught!
    """
    context_text = "\n\n".join(contexts)
    
    # Step 1: Extract claims
    extraction_prompt = f"""Extract all distinct factual claims from this answer as a JSON list.
Only extract claims, not questions or instructions.

Answer: {answer}

Respond with ONLY a JSON array of strings, e.g. ["claim 1", "claim 2"]"""
    
    extract_response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": extraction_prompt}]
    )
    claims_raw = extract_response.content[0].text.strip()
    if claims_raw.startswith("```"):
        claims_raw = claims_raw.split("\n", 1)[1].rsplit("```", 1)[0]
    claims = json.loads(claims_raw)
    
    # Step 2: Verify each claim against context
    verification_prompt = f"""Given this context:

{context_text}

For each claim below, determine if it is SUPPORTED, CONTRADICTED, or NOT_MENTIONED by the context.

Claims:
{json.dumps(claims, indent=2)}

Respond with a JSON object mapping each claim to its status:
{{"claim text": "SUPPORTED" or "CONTRADICTED" or "NOT_MENTIONED"}}"""
    
    verify_response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": verification_prompt}]
    )
    verif_raw = verify_response.content[0].text.strip()
    if verif_raw.startswith("```"):
        verif_raw = verif_raw.split("\n", 1)[1].rsplit("```", 1)[0]
    verification = json.loads(verif_raw)
    
    # Step 3: Compute score
    supported = sum(1 for v in verification.values() if v == "SUPPORTED")
    faithfulness_score = supported / len(claims) if claims else 1.0
    
    return {
        "faithfulness_score": round(faithfulness_score, 3),
        "total_claims": len(claims),
        "supported_claims": supported,
        "claims": claims,
        "verification": verification
    }


# Test it
test_question = "What are vector embeddings?"
test_answer, test_contexts = rag_answer(test_question)

print("🔍 Testing faithfulness checker...")
print(f"\nAnswer:\n{test_answer}")
print(f"\nContexts retrieved: {len(test_contexts)}")

faith_result = check_faithfulness(test_answer, test_contexts)
print(f"\n📊 Faithfulness Results:")
print(f"  Score: {faith_result['faithfulness_score']:.3f} ({faith_result['supported_claims']}/{faith_result['total_claims']} claims supported)")
print("\n  Claim verification:")
for claim, status in faith_result['verification'].items():
    icon = "✅" if status == "SUPPORTED" else "❌" if status == "CONTRADICTED" else "⚠️"
    print(f"    {icon} [{status}] {claim[:80]}")

---
# 🏆 Part 5: Capstone — Full Eval Report with CI/CD Gate

Now we put it all together: build a complete eval report that includes:
- LLM-Judge scores (accuracy, completeness, clarity)
- RAGAS-style faithfulness
- A CI/CD gate that fails if scores are too low

This is what you'd run in GitHub Actions before every deployment of your AutoResearcher agent.

In [ ]:
from datetime import datetime

def run_full_eval_report(
    dataset: list,
    rag_fn: Callable,
    quality_threshold: float = 7.0,
    faithfulness_threshold: float = 0.7
) -> dict:
    """
    Complete eval pipeline that would run in CI/CD.
    Returns a report dict + sets exit code (pass/fail).
    """
    print("🔬 Running Full Eval Suite...")
    print("=" * 60)
    
    case_results = []
    
    for case in dataset:
        print(f"\n📋 [{case['id']}] {case['question'][:70]}")
        
        # Get RAG answer
        answer, contexts = rag_fn(case["question"])
        
        # LLM Judge
        judge_scores = llm_judge_pointwise(case["question"], answer, {
            "accuracy": "Is the answer factually correct?",
            "completeness": "Does the answer fully address the question?",
            "clarity": "Is the answer clear and well-structured?"
        })
        avg_quality = sum(v["score"] for v in judge_scores.values()) / len(judge_scores)
        
        # Faithfulness
        faith = check_faithfulness(answer, contexts)
        
        quality_pass = avg_quality >= quality_threshold
        faith_pass = faith["faithfulness_score"] >= faithfulness_threshold
        overall_pass = quality_pass and faith_pass
        
        print(f"  Quality score: {avg_quality:.1f}/10 {'✅' if quality_pass else '❌'}")
        print(f"  Faithfulness:  {faith['faithfulness_score']:.3f} {'✅' if faith_pass else '❌'}")
        print(f"  Overall:       {'PASS ✅' if overall_pass else 'FAIL ❌'}")
        
        case_results.append({
            "id": case["id"],
            "quality_score": round(avg_quality, 2),
            "faithfulness": faith["faithfulness_score"],
            "quality_pass": quality_pass,
            "faith_pass": faith_pass,
            "overall_pass": overall_pass
        })
        
        time.sleep(0.5)
    
    # Aggregate
    total = len(case_results)
    passed = sum(1 for r in case_results if r["overall_pass"])
    pass_rate = passed / total
    avg_quality = sum(r["quality_score"] for r in case_results) / total
    avg_faithfulness = sum(r["faithfulness"] for r in case_results) / total
    
    # CI/CD gate — fail if < 80% pass rate
    ci_pass = pass_rate >= 0.8
    
    report = {
        "timestamp": datetime.now().isoformat(),
        "total_cases": total,
        "passed": passed,
        "pass_rate": pass_rate,
        "avg_quality_score": round(avg_quality, 2),
        "avg_faithfulness": round(avg_faithfulness, 3),
        "ci_gate_passed": ci_pass,
        "case_results": case_results
    }
    
    print("\n" + "=" * 60)
    print("📊 EVAL SUITE SUMMARY")
    print("=" * 60)
    print(f"  Cases:              {passed}/{total} passed")
    print(f"  Pass rate:          {pass_rate:.0%}")
    print(f"  Avg quality score:  {avg_quality:.2f}/10")
    print(f"  Avg faithfulness:   {avg_faithfulness:.3f}")
    print(f"  CI/CD Gate:         {'✅ PASS — SAFE TO DEPLOY' if ci_pass else '❌ FAIL — BLOCK DEPLOY'}")
    
    return report


# Run it!
final_report = run_full_eval_report(
    dataset=ragas_test_cases,
    rag_fn=rag_answer
)

---
## 🔄 Part 6: Wiring Evals into GitHub Actions CI/CD

Your eval script needs to return a non-zero exit code on failure so CI blocks the deployment.

Here's how you'd add this to your AutoResearcher repo's CI pipeline:

```yaml
# .github/workflows/eval.yml
name: Eval Gate

on:
  pull_request:
    branches: [main]

jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - name: Run eval suite
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: python evals/run_evals.py  # exits with code 1 if below threshold
```

And your `evals/run_evals.py` ends with:

```python
import sys
report = run_full_eval_report(dataset, rag_fn)
sys.exit(0 if report["ci_gate_passed"] else 1)  # 1 = fail = block PR merge
```

This means **no one can merge a prompt or code change that degrades quality below your threshold.** 🔒

In [ ]:
import sys

# Simulate what your CI/CD script would do
print("🔄 Simulating CI/CD exit behavior...")

if final_report["ci_gate_passed"]:
    print("✅ CI gate PASSED — would exit 0 (allow deployment)")
    # sys.exit(0)  # Uncomment in real CI script
else:
    print("❌ CI gate FAILED — would exit 1 (block deployment)")
    # sys.exit(1)  # Uncomment in real CI script

print("\n💡 EXPERIMENT ideas:")
print("  1. Lower the pass_threshold to 5.0 and see if more cases pass")
print("  2. Add a new eval case about a topic NOT in the knowledge base")
print("     (tests what happens when retrieval fails — low faithfulness expected)")
print("  3. Add a 'hallucination injection' — modify the answer to include a made-up fact")
print("     and verify the faithfulness checker catches it")
print("  4. Run the poor-quality system through the full report — verify the CI gate blocks it")

---
# 📚 Lesson 17 Summary

## What You Built Today

| Component | What it does |
|-----------|--------------|
| `llm_judge_pointwise()` | Scores any AI output on custom criteria (1-10) |
| `llm_judge_pairwise()` | Compares two outputs and picks the better one |
| `run_eval_pipeline()` | Runs pointwise judge over a full dataset, detects regressions |
| RAGAS integration | 4-metric RAG evaluation (faithfulness, relevancy, recall, precision) |
| `check_faithfulness()` | Manual faithfulness check via claim extraction + verification |
| `run_full_eval_report()` | Complete eval suite with CI/CD gate (pass/fail) |

## Key Mental Models

1. **Evals = unit tests for AI behavior** — run them on every change, gate deployments on them
2. **LLM judges scale human judgment** — use a cheap fast model (Haiku) as judge, reserve expensive model for generation
3. **RAGAS decomposes RAG quality** into 4 orthogonal signals — low faithfulness and low recall have different root causes and different fixes
4. **Pairwise > Pointwise** for A/B testing — avoids calibration drift between judge runs

## What's Next

**Lesson 18 → AI Security** — prompt injection attacks, guardrails, red-teaming your AutoResearcher agent, defending against jailbreaks, PII detection.

This is especially relevant now that you have a deployed FastAPI service (Lesson 16) — you'll learn how to protect it.

---
*Lesson 17 of 23 — Phase 3: Production AI Engineering*